In [1]:
# Libraries for parsing data
import os
import pandas as pd
import lxml.etree as etree
import pickle

In [2]:
folder_path = "C:\\Users\\Amit\\Google Drive\\Study\\Law\\SLS\\Working Papers\\evidence in legislatures_Barak-Corren&Ash\\hearings_testing"

## Load and Sample Data

Depending on the size and vocabulary of the input dataset, runtime of this script may vary. To process the entire dataset, set `sample_size` to `len(input_files)`. Larger datasets can be run on the multiprocessing version of this script.

In [4]:
# Set corpus to the folder of files you want to use
os.chdir('/Users/Amit/Documents/congressional_hearings_ash&corren/data')
# save the directory path to a variable
corpus = os.getcwd()
# get a list of the folders in this directory; ignore the .DS_Store folder
folders = [f.path for f in os.scandir(corpus) if f.is_dir()]
folders

['C:\\Users\\Amit\\Documents\\congressional_hearings_ash&corren\\data\\2010s']

In [5]:
# get a list of the files in each folder.
input_files = []
for folder in folders:
    for file in os.listdir(folder):
        input_files.append(folder + '/' + file)

In [7]:
# get a specific hearing file
hearing_num = 'hrg-2011-nar-0010' # copied from a hearing record in Congressional
hearing_file = ''
for file in input_files:
    if (hearing_num.lower() + '.xml') in file:
        hearing_file = file

In [69]:
'''
# for testing functions on a smaller subset of files
# Select the number of articles to sample
import random

sample_size = 5000

# Generate a sample of articles
try:
    sample_input_files = random.sample(input_files, sample_size)

except ValueError:
    sample_input_files = input_files
    
print("Currently sampling", len(sample_input_files), "documents.")
'''

Currently sampling 5000 documents.


In [40]:
'''
# inspect a sample file
import lxml.etree as etree

# Increase the size limit for processing text
parser = etree.XMLParser(huge_tree=True)

# Parse the XML file with the custom parser
tree = etree.parse(os.path.join(hearing_file), parser)
root = tree.getroot()


# Find the <Text> tag using XPath
fulltext = root.find('.//fulltext')

# Remove the <Text> tag and its contents
if fulltext is not None:
    fulltext.getparent().remove(fulltext)

# Print the modified XML document
newtree = etree.tostring(root, encoding='unicode')
print(newtree)
'''


"\n# Find the <Text> tag using XPath\nfulltext = root.find('.//fulltext')\n\n# Remove the <Text> tag and its contents\nif fulltext is not None:\n    fulltext.getparent().remove(fulltext)\n\n# Print the modified XML document\nnewtree = etree.tostring(root, encoding='unicode')\nprint(newtree)\n"

## Gather Metadata Fields

This section will gather text fields from the articles and add them to lists that will be used to make a dataframe.

In [8]:
# function to extract relevant biographical information from the participant element and store it in a dictionary
def get_bio_info(participant_element, participant_type):
    # define the namespace for committees and parties & states tags
    nsmap = {'gis': 'http://proquest.com/cosmos/gis', 'rc': 'http://proquest.com/cosmos/rc'}

    participant_dict = {
            f'{participant_type.lower()}_num': None,
            f'{participant_type.lower()}_fname': None,
            f'{participant_type.lower()}_honorific': None,
            f'{participant_type.lower()}_lname': participant_element.find('.//last-name').text,
            f'{participant_type.lower()}_role': None,
            f'{participant_type.lower()}_affiliation': '',
            f'{participant_type.lower()}_party': None,
            f'{participant_type.lower()}_state': None
        }

    first_name_element = participant_element.find('.//first-name')
    honorific_element = participant_element.find('.//honorific')
    position_element = participant_element.find('.//position')
    party_element = participant_element.find('./congressperson/rc:party', namespaces=nsmap)
    state_element = participant_element.find('./congressperson/rc:state', namespaces=nsmap)
    personal_role_element = participant_element.find('.//personal-role')
    org_elements = participant_element.findall('.//organization-term')
    text_element = participant_element.find('.//text')

    if first_name_element is not None:
        participant_dict[f'{participant_type.lower()}_fname'] = first_name_element.text

    if honorific_element is not None:
        participant_dict[f'{participant_type.lower()}_honorific'] = honorific_element.text

    if position_element is not None:
        participant_dict[f'{participant_type.lower()}_role'] = position_element.text
        if party_element is not None:
            participant_dict[f'{participant_type.lower()}_party'] = party_element.get('code')
        if state_element is not None:
            participant_dict[f'{participant_type.lower()}_state'] = state_element.get('code')

    elif personal_role_element is not None:
        # get any text from the tag including any text in <em> or <b> tags and add a space after any <em> or <b> tag text
        participant_dict[f'{participant_type.lower()}_role'] = ' '.join([text.strip() for text in personal_role_element.itertext()])
        # if either ', p.' or ' p.' are in the personal_role_element.text, split on them and take only the first part
        if ', p.' in participant_dict[f'{participant_type.lower()}_role']:
            participant_dict[f'{participant_type.lower()}_role'] = participant_dict[f'{participant_type.lower()}_role'].split(', p.', maxsplit=1)[0].strip()
        elif 'p.' in participant_dict[f'{participant_type.lower()}_role']:
            participant_dict[f'{participant_type.lower()}_role'] = participant_dict[f'{participant_type.lower()}_role'].split('p.', maxsplit=1)[0].strip()
        
        # add an affiliation if there is one
        if org_elements is not None:
            participant_dict[f'{participant_type.lower()}_affiliation'] = ', '.join(org.text for org in org_elements if org.text is not None)

    elif text_element is not None:
        # get any text from the .//text tag including any text in <em> or <b> tags and add a space after any <em> or <b> tag text
        participant_dict[f'{participant_type.lower()}_affiliation'] = ' '.join([text.strip() for text in text_element.itertext()])
        # if ',' is in the affiliation, split on it and take the first part as the role
        if ',' in participant_dict[f'{participant_type.lower()}_affiliation']:
            role_affiliation_parts = participant_dict[f'{participant_type.lower()}_affiliation'].split(',', maxsplit=1)
            if len(role_affiliation_parts) > 1:
                participant_dict[f'{participant_type.lower()}_role'] = role_affiliation_parts[0].strip()
                participant_dict[f'{participant_type.lower()}_affiliation'] = role_affiliation_parts[1].strip()
        # if either ', p.' or ' p.' are in the affiliation, split on them and take only the first part
        if ', p.' in participant_dict[f'{participant_type.lower()}_affiliation']:
            participant_dict[f'{participant_type.lower()}_affiliation'] = participant_dict[f'{participant_type.lower()}_affiliation'].split(', p.', maxsplit=1)[0].strip()
        elif 'p.' in participant_dict[f'{participant_type.lower()}_affiliation']:
            participant_dict[f'{participant_type.lower()}_affiliation'] = participant_dict[f'{participant_type.lower()}_affiliation'].split('p.', maxsplit=1)[0].strip()

    return participant_dict

In [9]:
# function to process the participant elements and return a list of dictionaries with the relevant biographical information
def process_participant_elements(participant_elements, p_type):
    participant_type = p_type
    result = []

    for num, participant_element in enumerate(participant_elements):
        participant_element_dict = get_bio_info(participant_element, participant_type)
        participant_element_dict[f'{participant_type.lower()}_num'] = num + 1
        result.append(participant_element_dict)

        # get the proxied witness if there is one in the participant_element, run the get_bio_info function on it, and add to the participant_element_dict
        proxy_for_element = participant_element.find('.//hearing-participant[@type="WITNESS-BY-PROXY"]')
        if proxy_for_element is not None:
            proxy_for_element_dict = get_bio_info(proxy_for_element, 'PROXY_FOR')
            participant_element_dict.update(proxy_for_element_dict)

    return result

In [10]:
import re

def roman_to_int(roman):
    roman_numerals = {'i': 1, 'v': 5, 'x': 10, 'l': 50, 'c': 100, 'd': 500, 'm': 1000}
    result = 0
    roman = roman.lower()
    i = 0
    while i < len(roman):
        if i + 1 < len(roman) and roman_numerals[roman[i]] < roman_numerals[roman[i + 1]]:
            result -= roman_numerals[roman[i]]
        else:
            result += roman_numerals[roman[i]]
        i += 1
    return result

def extract_page_info(page_string):
    parts = re.split(r',\s+', page_string)
    roman_value_sum = 0
    total_page_count = 0

    pattern = re.compile(r'(?:(?P<roman>[ivxlcdm]+)\+)?(?P<start>\d{1,4}(?:,\d{3})?)(?:-(?P<end>\d{1,4}(?:,\d{3})?))?|(?P<single>\d{1,3}(?:,\d{3})?)\s*p\.', re.IGNORECASE) 

    for part in parts:
        match = pattern.search(part)
        if match:
            if match.group('single'):
                single_page_count = int(match.group('single').replace(',', ''))
                total_page_count += single_page_count
            else:
                if match.group('roman'):
                    roman_value = roman_to_int(match.group('roman'))
                    roman_value_sum += roman_value

                if match.group('end'):
                    start = int(match.group('start').replace(',', ''))
                    end = int(match.group('end').replace(',', ''))
                    range_length = end - start + 1
                    total_page_count += range_length
                else:  # This is a single range count from 1 to 'start'
                    start = int(match.group('start').replace(',', ''))
                    total_page_count += start  # Corrected to count all pages from 1 to 'start'

    return roman_value_sum + total_page_count

# Testing the function with the updated regex and examples
print(extract_page_info("vi+1155-2290"))          # Roman numeral with range
print(extract_page_info("1,297 p."))             # Page number with commas
print(extract_page_info("iii+1-28, 49-186"))     # Multiple ranges with a Roman numeral


1142
1297
169


In [11]:
# Retrieve metadata from XML document
def getxmlcontent(file):
    # define the namespace for committees and parties & states tags
    nsmap = {'gis': 'http://proquest.com/cosmos/gis', 'rc': 'http://proquest.com/cosmos/rc'}
    try:
        # Increase the size limit for processing text as some files are very large
        parser = etree.XMLParser(huge_tree=True)
        tree = etree.parse(file, parser)
        root = tree.getroot()

        if root.find('.//accession-no') is not None:
            accession_no = root.find('.//accession-no').text
        else:
            accession_no = None

        if root.find('.//title') is not None:
            title = root.find('.//title').text
        else:
            title = None

        if root.find('.//url') is not None:
            url = root.find('.//url').text
        else:
            url = None

        if root.find('.//date') is not None:
            date = root.find('.//date').text
        else:
            date = None
            
        # get the text of <gis:congress> tag and save as congress_num
        if root.find('.//gis:congress', namespaces=nsmap) is not None:
            congress_num = int(root.find('.//gis:congress', namespaces=nsmap).text)
        else:
            congress_num = None

        # get the administration text from the  <gis:administration> tag
        if root.find('.//gis:administration', namespaces=nsmap) is not None:
            administration = root.find('.//gis:administration', namespaces=nsmap).text
        else:
            administration = None

        if  root.find('.//gis:committee-source', namespaces=nsmap) is not None:
            committee_source = root.findall('.//gis:committee-source', namespaces=nsmap)[-1].text
            if 'Subcommittee' in committee_source:
                committee_name = 'Committee' + committee_source.split(' Committee')[1]
                subcommittee_name = committee_source.split(' Committee')[0]
            else:
                committee_name = committee_source
                subcommittee_name = None
        # if there is no <gis:committee-source> tag, check for a <gis:agency-source> tag
        elif root.find('.//gis:agency-source', namespaces=nsmap) is not None:
            committee_name = root.findall('.//gis:agency-source', namespaces=nsmap)[-1].text
            subcommittee_name = None
        else:
            committee_name = None
            subcommittee_name = None
      
        if root.find('.//gis:subject', namespaces=nsmap) is not None:
            subject_element = root.findall('.//gis:subject', namespaces=nsmap)
            subjects = [subject.text for subject in subject_element]
        else:
            subjects = None
        
        # get the text of all the <p> tags in the <display> tag as the abstract. If there's no <p> tag, get the text of the <noc> tag. If there are any <em> or <b> tags in the abstract, remove them
        if root.find('.//display') is not None:
            display_element = root.find('.//display')
            if display_element.findall('.//p'):
                abstract = ' '.join([text.strip() for text in display_element.findall('.//p')[-1].itertext()])
            elif display_element.find('.//noc') is not None:
                abstract = display_element.find('.//noc').text
            else:
                abstract = None
        else:
            abstract = None
        
        # get the text of the <pdf-page-count> tag or the <length> tag
        if root.find('.//pdf-page-count') is not None:
            # get the text of the <pdf-page-count> tag and store as an integer; replace commas if there are any. If the tag is present but has no text, get page_count from the length tag
            page_count = int(root.find('.//pdf-page-count').text.replace(",", "")) if root.find('.//pdf-page-count').text is not None else extract_page_info(root.find('.//length').text)
        # if no <pdf-page-count> tag, get the number from the <length> tag and run the extract_page_info function
        elif root.find('.//length') is not None:
            page_count = extract_page_info(root.find('.//length').text)        
        else:
            page_count = None
        
        if root.find('.//hearing-participant[@type="NOMINEE"]') is not None:
            nominees_element = root.findall('.//hearing-participant[@type="NOMINEE"]')
            nominees = process_participant_elements(nominees_element, 'NOMINEE')

        else:
            nominees = None

        if root.find('.//hearing-participant[@type="ON-BEHALF-OF"]') is not None:
            proxies_element = root.findall('.//hearing-participant[@type="ON-BEHALF-OF"]')
            proxies = process_participant_elements(proxies_element, 'PROXY_WITNESS')

        else:
            proxies = None

        if root.find('.//hearing-participant[@type="WITNESS"]') is not None:
            witnesses_element = root.findall('.//hearing-participant[@type="WITNESS"]')
            witnesses = process_participant_elements(witnesses_element, 'WITNESS')

        else:
            witnesses = None
        
        if root.find('.//billtext') is not None:
            billnum = root.find('.//billtext').text
        else:
            billnum = None

    except Exception as e:
        print(f"Error while parsing file {file}: {e}")

    return accession_no, title, url, date, congress_num, administration, abstract, page_count, committee_name, subcommittee_name, subjects, billnum, nominees, proxies, witnesses

In [12]:
# Create lists to store article IDs, titles, dates, and text
accession_no_list = []
title_list = []
url_list = []
date_list = []
congress_num_list = []
administration_list = []
abstracts_list = []
page_count_list = []
committee_list = []
subcommittee_list = []
subjects_list = []
billnum_list = []
nominees_list = []
proxies_list = []
witnesses_list = []

In [13]:
# Parse files and add data to lists
#for file in sample_input_files:
for file in input_files:
    # Retrieve the metadata
    accession_no, title, url, date, congress_num, administration, abstract, page_count, committee_name, subcommittee_name, subjects, billnum, nominees, proxies, witnesses = getxmlcontent(file)

    # Store metadata to lists
    accession_no_list.append(accession_no)
    title_list.append(title)
    url_list.append(url)
    date_list.append(date)
    congress_num_list.append(congress_num)
    administration_list.append(administration)
    abstracts_list.append(abstract)
    page_count_list.append(page_count)
    committee_list.append(committee_name)
    subcommittee_list.append(subcommittee_name)
    subjects_list.append(subjects)
    billnum_list.append(billnum)
    nominees_list.append(nominees)
    proxies_list.append(proxies)
    witnesses_list.append(witnesses)


## Create Dataframe

This section uses the collected fields to make a dataframe.

In [14]:
# Create a list of lists to store the lists made above
hearings_lists = [accession_no_list, title_list, url_list, date_list, congress_num_list,  administration_list, abstracts_list, page_count_list, committee_list, subcommittee_list, subjects_list, billnum_list, nominees_list, proxies_list, witnesses_list] 

In [15]:
# Save the set of lists to a file
with open("Users/Amit/Documents/Google Drive/Study/Law/SLS/Working Papers/evidence in legislatures_Barak-Corren&Ash/scripts/hearings_lists.pickle", "wb") as file:
    pickle.dump(hearings_lists, file)

FileNotFoundError: [Errno 2] No such file or directory: 'Users/Amit/Documents/Google Drive/Study/Law/SLS/Working Papers/evidence in legislatures_Barak-Corren&Ash/scripts/hearings_lists.pickle'

In [16]:
# Create a dataframe, setting each of the columns to one of the lists made in the cell above except for the people lists
df_main = pd.DataFrame({
    'Accession_No': accession_no_list,
    'Title': title_list,
    'URL': url_list,
    'Date': date_list,
    'Congress_Num': congress_num_list,
    'Administration': administration_list,
    'Abstract': abstracts_list,
    'Page_Count': page_count_list,
    'Committee': committee_list,
    'Subcommittee': subcommittee_list,
    'billnum': billnum_list,
    'Subjects': subjects_list
})


In [18]:
# create a dataframe for the witnesses, nominees, and proxies
df_witnesses = pd.DataFrame({
    'Accession_No': accession_no_list,
    'Witnesses': witnesses_list
})
df_nominees = pd.DataFrame({
    'Accession_No': accession_no_list,
    'Nominees': nominees_list
})
df_proxies = pd.DataFrame({
    'Accession_No': accession_no_list,
    'Proxies': proxies_list
})

In [19]:
# get the total page count for the df by summing the Page_Count column. Drop any commas in the string and convert from str to int first. Remove any null values
df_main['Page_Count'].sum() # 1,000,000

2263848

In [20]:
# find any null values in the Committee column
df_main[df_main['Committee'].isnull()]

,Accession_No,Title,URL,Date,Congress_Num,Administration,Abstract,Page_Count,Committee,Subcommittee,billnum,Subjects
222,HRG-2010-COP-0001,"Commercial Real Estate. Congressional Hearing,...",http://search.proquest.com/congressional/view/...,2010-01-27,111,Barack Obama (2009-2017),Atlanta Mayor Kasim Reed presents opening rema...,150,None,None,None,"[Business cycles, Real estate, Loans, Banking,..."
223,HRG-2010-COP-0002,Citigroup and the Troubled Asset Relief Progra...,http://search.proquest.com/congressional/view/...,2010-03-04,111,Barack Obama (2009-2017),Supplementary material (p. 93-115) includes a ...,121,None,None,None,"[Business cycles, Fiscal policy, Business gove..."
224,HRG-2010-COP-0003,GMAC Financial Services and the Troubled Asset...,http://search.proquest.com/congressional/view/...,2010-02-25,111,Barack Obama (2009-2017),Hearing before the Congressional Oversight Pan...,104,None,None,None,"[Automobiles, Automobile industry, Business cy..."
225,HRG-2010-COP-0004,"Small Business Lending. Congressional Hearing,...",http://search.proquest.com/congressional/view/...,2010-04-27,111,Barack Obama (2009-2017),"Hearing in Phoenix, Ariz., before the Congress...",104,None,None,None,"[Business cycles, Credit, Capital formation, S..."
226,HRG-2010-COP-0005,Hearing with Secretary Timothy Geithner. Congr...,http://search.proquest.com/congressional/view/...,2010-06-22,111,Barack Obama (2009-2017),Supplementary material (p. 64-77) includes wi...,83,None,None,None,"[Business cycles, Economic policy, Fiscal poli..."
227,HRG-2010-COP-0006,TARP and Other Government Assistance for AIG. ...,http://search.proquest.com/congressional/view/...,2010-05-26,111,Barack Obama (2009-2017),Supplementary material (p. 231-235) includes ...,241,None,None,None,"[Government spending, Regulation of financial ..."
228,HRG-2010-COP-0007,TARP and Executive Compensation Restrictions. ...,http://search.proquest.com/congressional/view/...,2010-10-21,111,Barack Obama (2009-2017),BACKGROUND AND CONTEXT: TARP authorized Depart...,108,None,None,None,"[Executives, Regulation of financial instituti..."
229,HRG-2010-COP-0008,TARP Foreclosure Mitigation Programs. Congress...,http://search.proquest.com/congressional/view/...,2010-10-27,111,Barack Obama (2009-2017),Supplementary material (p. 172-188) includes ...,194,None,None,None,"[Foreclosures, Home financing, Housing prices,..."
230,HRG-2010-COP-0009,Treasury's Use of Contracting Authority Under ...,http://search.proquest.com/congressional/view/...,2010-09-22,111,Barack Obama (2009-2017),BACKGROUND AND CONTEXT: Many of TARP's most cr...,127,None,None,None,"[Fiscal policy, Economic policy, Regulation of..."
231,HRG-2010-COP-0010,Hearing with Treasury Secretary Geithner. Cong...,http://search.proquest.com/congressional/view/...,2010-12-16,111,Barack Obama (2009-2017),Supplementary material (p. 77-82) includes a ...,88,None,None,None,"[Business cycles, Economic policy, Fiscal poli..."


## Save Outputs

In [42]:
# save df_main to parquet
df_main.to_parquet(
    '/Users/darringtonj/Dropbox (Princeton)/princeton/pythonScripts/Congressional_Hearings/my_output_files/hearings_main.parquet')
# save df_witnesses to json
df_witnesses.to_json(
    '/Users/darringtonj/Dropbox (Princeton)/princeton/pythonScripts/Congressional_Hearings/my_output_files/witnesses.json',
    orient='records')
# save df_nominees to json
df_nominees.to_json(
    '/Users/darringtonj/Dropbox (Princeton)/princeton/pythonScripts/Congressional_Hearings/my_output_files/nominees.json',
    orient='records')
# save df_proxies to json
df_proxies.to_json(
    '/Users/darringtonj/Dropbox (Princeton)/princeton/pythonScripts/Congressional_Hearings/my_output_files/proxies.json',
    orient='records')

## Flatten data on the people lists

In [8]:
# drop all rows where the 'Nominees' column is null
df_nominees = df_nominees.dropna(subset=['Nominees'])

In [9]:
nominees_flattened_data = []
primary_key = 1

for accession_id, nominees in df_nominees['Nominees'].items():
    for nominee in nominees:
        nominee_record = {
            'pk': primary_key,
            'Accession_No': df_nominees['Accession_No'][accession_id],
            **nominee
        }
        nominees_flattened_data.append(nominee_record)
        primary_key += 1

# Preview the first few rows of the flattened data
nominees_flattened_data[:10]

[{'pk': 1,
  'Accession_No': 'HRG-1884-PPR-0006',
  'nominee_num': 1,
  'nominee_fname': 'George F.',
  'nominee_honorific': None,
  'nominee_lname': 'Evans',
  'nominee_role': 'Postmaster, Martinsburg, WVa',
  'nominee_affiliation': '',
  'nominee_party': None,
  'nominee_state': None},
 {'pk': 2,
  'Accession_No': 'HRG-2009-CST-0041',
  'nominee_num': 1,
  'nominee_fname': 'Philip E.',
  'nominee_honorific': None,
  'nominee_lname': 'Coyle',
  'nominee_role': 'Associate Director, National Security and International Affairs',
  'nominee_affiliation': 'Office of Science and Technology Policy',
  'nominee_party': None,
  'nominee_state': None},
 {'pk': 3,
  'Accession_No': 'HRG-2009-CST-0041',
  'nominee_num': 2,
  'nominee_fname': 'Scott B.',
  'nominee_honorific': None,
  'nominee_lname': 'Quehl',
  'nominee_role': 'Chief Financial Officer and Assistant Secretary, Administration',
  'nominee_affiliation': 'Department of Commerce',
  'nominee_party': None,
  'nominee_state': None},
 {'

In [25]:
# Create a dataframe from the flattened data
df_nominees_flattened = pd.DataFrame(nominees_flattened_data)

In [12]:
# drop all rows where the 'Witnesses' column is null
df_witnesses = df_witnesses.dropna(subset=['Witnesses'])

In [13]:
witnesses_flattened_data = []
primary_key = 1

for accession_id, witnesses in df_witnesses['Witnesses'].items():
    for witness in witnesses:
        witness_record = {
            'pk': primary_key,
            'Accession_No': df_witnesses['Accession_No'][accession_id],
            **witness
        }
        witnesses_flattened_data.append(witness_record)
        primary_key += 1

In [14]:
# Preview the first few rows of the flattened data
witnesses_flattened_data[83000:83010]

[{'pk': 83001,
  'Accession_No': 'HRG-2002-SAS-0019',
  'witness_num': 35,
  'witness_fname': 'Susan M.',
  'witness_honorific': None,
  'witness_lname': 'Schwartz',
  'witness_role': 'Deputy Director',
  'witness_affiliation': 'Government Relations/Health Affairs, Retired Officers Association; all five representing Military Coalition.',
  'witness_party': None,
  'witness_state': None},
 {'pk': 83002,
  'Accession_No': 'HRG-2001-SCI-0048',
  'witness_num': 1,
  'witness_fname': 'Stephen D.',
  'witness_honorific': None,
  'witness_lname': 'Ansolabehere',
  'witness_role': 'Professor',
  'witness_affiliation': 'Political Science, MIT.',
  'witness_party': None,
  'witness_state': None},
 {'pk': 83003,
  'Accession_No': 'HRG-2001-SCI-0048',
  'witness_num': 2,
  'witness_fname': 'Rebecca T.',
  'witness_honorific': None,
  'witness_lname': 'Mercuri',
  'witness_role': 'Assistant Professor',
  'witness_affiliation': 'Computer Science, Bryn Mawr College.',
  'witness_party': None,
  'witn

In [15]:
# Get all keys in the dictionaries
all_keys = set().union(*(d.keys() for d in witnesses_flattened_data))

print(all_keys)

{'witness_affiliation', 'witness_party', 'witness_fname', 'proxy_for_affiliation', 'pk', 'proxy_for_num', 'witness_lname', 'proxy_for_fname', 'proxy_for_honorific', 'witness_state', 'witness_role', 'proxy_for_state', 'proxy_for_role', 'witness_honorific', 'proxy_for_party', 'Accession_No', 'witness_num', 'proxy_for_lname'}


In [16]:
# get all the index of the items in witnesses_flattened_data where the 'proxy_for_lname' key is not null. These should have been captured in the 'proxies' data
indices = [i for i, d in enumerate(witnesses_flattened_data) if 'proxy_for_lname' in d]
print(indices)


[336821, 1021334, 1043219]


In [17]:
# get these items [336821, 1021334, 1043219]
print([witnesses_flattened_data[i] for i in indices])

[{'pk': 336822, 'Accession_No': 'HRG-1981-HAP-0028', 'witness_num': 26, 'witness_fname': 'Jack', 'witness_honorific': None, 'witness_lname': 'Edwards', 'witness_role': 'Representative', 'witness_affiliation': '', 'witness_party': 'R', 'witness_state': 'AL', 'proxy_for_num': None, 'proxy_for_fname': 'Sheldon L.', 'proxy_for_honorific': None, 'proxy_for_lname': 'Morgan', 'proxy_for_role': 'pres', 'proxy_for_affiliation': 'Warrior-Tombigbee Dev Assn', 'proxy_for_party': None, 'proxy_for_state': None}, {'pk': 1021335, 'Accession_No': 'HRG-1977-SAP-0001', 'witness_num': 114, 'witness_fname': 'Quentin N.', 'witness_honorific': None, 'witness_lname': 'Burdick', 'witness_role': 'Senator', 'witness_affiliation': '', 'witness_party': 'D', 'witness_state': 'ND', 'proxy_for_num': None, 'proxy_for_fname': 'Arthur A.', 'proxy_for_honorific': None, 'proxy_for_lname': 'Link', 'proxy_for_role': 'Gov', 'proxy_for_affiliation': 'NDak', 'proxy_for_party': None, 'proxy_for_state': None}, {'pk': 1043220, 'A

In [23]:
# Create a dataframe from the flattened data
df_witnesses_flattened = pd.DataFrame(witnesses_flattened_data)

In [19]:
# drop all rows where the 'Proxies' column is null
df_proxies = df_proxies.dropna(subset=['Proxies'])

In [20]:
proxies_flattened_data = []
primary_key = 1

for accession_id, proxies in df_proxies['Proxies'].items():
    for proxy in proxies:
        proxy_record = {
            'pk': primary_key,
            'Accession_No': df_proxies['Accession_No'][accession_id],
            **proxy
        }
        proxies_flattened_data.append(proxy_record)
        primary_key += 1

# Preview the first few rows of the flattened data
proxies_flattened_data[:10]

[{'pk': 1,
  'Accession_No': 'HRG-2003-HRC-0031',
  'proxy_witness_num': 1,
  'proxy_witness_fname': 'Freeman K.',
  'proxy_witness_honorific': None,
  'proxy_witness_lname': 'Johnson',
  'proxy_witness_role': 'Assistant Director',
  'proxy_witness_affiliation': 'Nevada Department of Conservation and Natural Resources; on behalf of:',
  'proxy_witness_party': None,
  'proxy_witness_state': None,
  'proxy_for_num': None,
  'proxy_for_fname': 'Kenny C.',
  'proxy_for_honorific': None,
  'proxy_for_lname': 'Guinn',
  'proxy_for_role': 'Governor',
  'proxy_for_affiliation': 'Nevada.',
  'proxy_for_party': None,
  'proxy_for_state': None},
 {'pk': 2,
  'Accession_No': 'HRG-2000-HAP-0075',
  'proxy_witness_num': 1,
  'proxy_witness_fname': 'Abby',
  'proxy_witness_honorific': None,
  'proxy_witness_lname': 'Smith',
  'proxy_witness_role': 'Director',
  'proxy_witness_affiliation': 'Programs, Council on Library and Information Resources; on behalf of:',
  'proxy_witness_party': None,
  'proxy

In [24]:
# Create a dataframe from the flattened data
df_proxies_flattened = pd.DataFrame(proxies_flattened_data)

In [26]:
# save df_nominees_flattened to parquet
df_nominees_flattened.to_parquet('/Users/darringtonj/Dropbox (Princeton)/princeton/pythonScripts/Congressional_Hearings/my_output_files/nominees_flattened.parquet')
# save df_witnesses_flattened to parquet
df_witnesses_flattened.to_parquet('/Users/darringtonj/Dropbox (Princeton)/princeton/pythonScripts/Congressional_Hearings/my_output_files/witnesses_flattened.parquet')
# save df_proxies_flattened to parquet
df_proxies_flattened.to_parquet('/Users/darringtonj/Dropbox (Princeton)/princeton/pythonScripts/Congressional_Hearings/my_output_files/proxies_flattened.parquet')